# Entity Sanity Checks

Validate ontology entity outputs and export all summary tables to CSV.


In [10]:
from pathlib import Path
import pandas as pd


In [11]:
BASE = Path('../../ingestion/tmp/entities')
EXPORT_DIR = Path('../../exploration/tmp/notebook_exports/02_entity_sanity_checks')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'geographic': BASE / 'geographic.csv',
    'economic': BASE / 'economic.csv',
    'apartment_market': BASE / 'apartment_market.csv',
}

missing = [name for name, path in PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing entity files: {missing}. Run build_entities first.')

df_geo = pd.read_csv(PATHS['geographic'])
df_econ = pd.read_csv(PATHS['economic'])
df_apt = pd.read_csv(PATHS['apartment_market'])

dfs = {'geographic': df_geo, 'economic': df_econ, 'apartment_market': df_apt}
print('Export dir:', EXPORT_DIR.resolve())


Export dir: /Users/ColeHoffman/dev/gis_phl/gis-phl/exploration/tmp/notebook_exports/02_entity_sanity_checks


In [12]:
table_summary = pd.DataFrame([{
    'entity_table': name,
    'rows': len(df),
    'cols': len(df.columns),
    'columns': ', '.join(df.columns),
} for name, df in dfs.items()])
table_summary.to_csv(EXPORT_DIR / 'table_summary.csv', index=False)
table_summary


,entity_table,rows,cols,columns
0,geographic,146,5,"entity_id, geography_type, name, county_fips, ..."
1,economic,431,5,"entity_id, geography_entity_id, period, unempl..."
2,apartment_market,18620,6,"entity_id, geography_entity_id, period, rent_i..."


In [13]:
null_rows = []
for table_name, df in dfs.items():
    for col in df.columns:
        null_rows.append({
            'entity_table': table_name,
            'column': col,
            'null_pct': round(df[col].isna().mean() * 100, 2),
            'n_null': int(df[col].isna().sum()),
        })
null_df = pd.DataFrame(null_rows).sort_values(['null_pct', 'entity_table', 'column'], ascending=[False, True, True])
null_df.to_csv(EXPORT_DIR / 'null_rates.csv', index=False)
null_df


,entity_table,column,null_pct,n_null
9,economic,inflation_rate,100.00,431
3,geographic,county_fips,100.00,146
4,geographic,state_fips,100.00,146
15,apartment_market,rent_growth_12m,9.45,1760
14,apartment_market,rent_growth_1m,1.17,217
13,apartment_market,rent_index,0.37,69
8,economic,unemployment_rate,0.23,1
10,apartment_market,entity_id,0.00,0
11,apartment_market,geography_entity_id,0.00,0
12,apartment_market,period,0.00,0


In [14]:
key_checks = []
for table_name, df in dfs.items():
    key_checks.append({
        'entity_table': table_name,
        'check_type': 'entity_id_uniqueness',
        'duplicate_count': int(df['entity_id'].duplicated().sum()),
        'unique_count': int(df['entity_id'].nunique()),
        'row_count': int(len(df)),
    })

if {'geography_entity_id', 'period'}.issubset(df_econ.columns):
    key_checks.append({
        'entity_table': 'economic',
        'check_type': 'geography_period_uniqueness',
        'duplicate_count': int(df_econ.duplicated(subset=['geography_entity_id', 'period']).sum()),
        'unique_count': int(df_econ[['geography_entity_id', 'period']].drop_duplicates().shape[0]),
        'row_count': int(len(df_econ)),
    })

if {'geography_entity_id', 'period'}.issubset(df_apt.columns):
    key_checks.append({
        'entity_table': 'apartment_market',
        'check_type': 'geography_period_uniqueness',
        'duplicate_count': int(df_apt.duplicated(subset=['geography_entity_id', 'period']).sum()),
        'unique_count': int(df_apt[['geography_entity_id', 'period']].drop_duplicates().shape[0]),
        'row_count': int(len(df_apt)),
    })

key_checks_df = pd.DataFrame(key_checks)
key_checks_df.to_csv(EXPORT_DIR / 'key_checks.csv', index=False)
key_checks_df


,entity_table,check_type,duplicate_count,unique_count,row_count
0,geographic,entity_id_uniqueness,0,146,146
1,economic,entity_id_uniqueness,0,431,431
2,apartment_market,entity_id_uniqueness,0,18620,18620
3,economic,geography_period_uniqueness,0,431,431
4,apartment_market,geography_period_uniqueness,0,18620,18620


In [15]:
period_cov = []
for table_name, df in [('economic', df_econ), ('apartment_market', df_apt)]:
    tmp = df.copy()
    tmp['period_dt'] = pd.to_datetime(tmp['period'], errors='coerce')
    period_cov.append({
        'entity_table': table_name,
        'min_period': tmp['period_dt'].min(),
        'max_period': tmp['period_dt'].max(),
        'n_unique_periods': int(tmp['period_dt'].nunique()),
        'n_invalid_periods': int(tmp['period_dt'].isna().sum()),
    })
period_cov_df = pd.DataFrame(period_cov)
period_cov_df.to_csv(EXPORT_DIR / 'period_coverage.csv', index=False)
period_cov_df


,entity_table,min_period,max_period,n_unique_periods,n_invalid_periods
0,economic,1990-01-01,2025-11-01,431,0
1,apartment_market,2015-01-31,2026-01-31,133,0


In [16]:
geo_ids = set(df_geo['entity_id'].dropna().astype(str))
econ_missing_geo = df_econ[~df_econ['geography_entity_id'].astype(str).isin(geo_ids)]
apt_missing_geo = df_apt[~df_apt['geography_entity_id'].astype(str).isin(geo_ids)]

ref_integrity_df = pd.DataFrame([
    {'entity_table': 'economic', 'rows_missing_geography_key': int(len(econ_missing_geo))},
    {'entity_table': 'apartment_market', 'rows_missing_geography_key': int(len(apt_missing_geo))},
])
ref_integrity_df.to_csv(EXPORT_DIR / 'referential_integrity.csv', index=False)
ref_integrity_df


,entity_table,rows_missing_geography_key
0,economic,0
1,apartment_market,0


In [17]:
econ_desc = df_econ['unemployment_rate'].describe().to_frame(name='unemployment_rate')
econ_desc.to_csv(EXPORT_DIR / 'economic_unemployment_describe.csv')
econ_desc


,unemployment_rate
count,430.000000
mean,5.599070
std,1.650782
min,3.600000
25%,4.300000
50%,5.100000
75%,6.600000
max,14.300000


In [18]:
apt_desc = df_apt[['rent_index', 'rent_growth_1m', 'rent_growth_12m']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
apt_desc.to_csv(EXPORT_DIR / 'apartment_market_describe.csv')
apt_desc


,rent_index,rent_growth_1m,rent_growth_12m
count,18551.000000,18403.000000,16860.000000
mean,1371.043039,0.004110,0.052676
std,479.857564,0.006166,0.040387
min,577.692850,-0.029645,-0.089938
1%,682.755464,-0.009464,-0.011126
5%,808.200967,-0.004991,0.007123
50%,1261.845590,0.003665,0.043597
95%,2354.296958,0.014784,0.133063
99%,2925.716850,0.022374,0.200007
max,3410.788299,0.046442,0.302511
